# CIG-Bench · Property demo

Run the pretrained rock-property model (GEM-style conditional paradigm) on a
3D seismic volume conditioned on sparse well-log values.

This notebook contains **no `cigvis` calls** — every figure is plain matplotlib.
For each section we show two side-by-side panels:

1. seismic in grayscale (left),
2. predicted property in `jet` (right).

Sections are sampled evenly along the inline / xline / time axes.

本 notebook 不使用 cigvis；每条剖面并排显示两栏 —— 左侧灰度地震，右侧
`jet` 配色的预测属性体。沿 inline / xline / time 三个方向各均匀抽 4 条
剖面。


## 1. Imports

In [ ]:
import numpy as np
import torch

from cig_bench.predictor.property import PropertyPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


## 2. Plot helper — side-by-side property sections

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_property_sections(seis, prop, axis, n_sec=4, prop_cmap="jet",
                           figsize_per_row=3.2, suptitle=None):
    """Two-panel sections side-by-side — seismic (gray) | property (jet).

    Parameters
    ----------
    seis, prop : (T, H, W) numpy arrays.
    axis       : 0 = inline (along H),
                 1 = xline  (along W),
                 2 = time   (along T).
    n_sec      : number of evenly-spaced sections (skips the two edges).
    """
    T, H, W = seis.shape
    if axis == 0:
        idx = np.linspace(0, H - 1, n_sec + 2, dtype=int)[1:-1]
        get = lambda v, i: v[:, i, :]
        ax_name = "inline"; x_lab, y_lab = "crossline", "depth"
    elif axis == 1:
        idx = np.linspace(0, W - 1, n_sec + 2, dtype=int)[1:-1]
        get = lambda v, i: v[:, :, i]
        ax_name = "xline"; x_lab, y_lab = "inline", "depth"
    elif axis == 2:
        idx = np.linspace(0, T - 1, n_sec + 2, dtype=int)[1:-1]
        get = lambda v, i: v[i, :, :]
        ax_name = "time"; x_lab, y_lab = "crossline", "inline"
    else:
        raise ValueError(f"axis must be 0/1/2, got {axis}")

    p_vmin, p_vmax = float(prop.min()), float(prop.max())

    fig, axes = plt.subplots(n_sec, 2, figsize=(8.5, figsize_per_row * n_sec))
    if n_sec == 1:
        axes = axes[None, :]

    for row, i in enumerate(idx):
        s = get(seis, i); p = get(prop, i)
        s_vmax = np.percentile(np.abs(s), 99)
        ax_s, ax_p = axes[row]

        ax_s.imshow(s, cmap="gray", aspect="auto", vmin=-s_vmax, vmax=s_vmax)
        ax_s.set_title(f"{ax_name} {i} — seismic", fontsize=10)

        ax_p.imshow(p, cmap=prop_cmap, aspect="auto", vmin=p_vmin, vmax=p_vmax)
        ax_p.set_title(f"{ax_name} {i} — property", fontsize=10)

        for ax in axes[row]:
            ax.set_xlabel(x_lab); ax.set_ylabel(y_lab)

    if suptitle:
        fig.suptitle(suptitle, fontsize=13, y=1.0)
    fig.tight_layout()
    plt.show()


## 3. Load seismic and sparse well-log property

The predictor expects two `(T, H, W)` volumes:

- `seis` — the seismic amplitude volume,
- `prop` — a property volume with **zeros everywhere except at well traces**,
  where the measured value of the target property (impedance, Vp, gamma ray,
  …) is stored.

模型期望两个 `(T, H, W)` 体：地震 + 稀疏井道属性（除井道位置外都是 0；井道
位置上是测得的属性值）。


In [ ]:
seis = np.load(r"../RealData/your_seis.npy").astype(np.float32)   # (T, H, W)
prop = np.load(r"../RealData/your_log.npy").astype(np.float32)    # (T, H, W)
assert seis.shape == prop.shape, (seis.shape, prop.shape)

print("seis shape :", seis.shape)
print("prop shape :", prop.shape,
      "  well voxels =", int((prop != 0).sum()))


## 4. Build the predictor and run inference

`PropertyPredictor()` loads the property HRNet weights from ModelScope on
first use.  Internally the predictor stacks three channels — seismic, sparse
property, well mask — and feeds them to the network.


In [ ]:
predictor = PropertyPredictor(device=device)         # auto-downloads weights

prop_volume, seis_used, well_info = predictor.predict(
    seis, prop,
    infer_shape=(640, 512, 512),
)
print("prop shape :", prop_volume.shape,
      "min/max:", float(prop_volume.min()), float(prop_volume.max()))
print("seis shape :", seis_used.shape)
print("wells       :", well_info["positions"].shape[0], "trace voxels")


## 5. Section visualisations

In [ ]:
plot_property_sections(seis_used, prop_volume, axis=0, n_sec=4,
                       suptitle="Property — inline sections")


In [ ]:
plot_property_sections(seis_used, prop_volume, axis=1, n_sec=4,
                       suptitle="Property — xline sections")


In [ ]:
plot_property_sections(seis_used, prop_volume, axis=2, n_sec=4,
                       suptitle="Property — time slices")
